# PEI Project — Sales Data Transformation Pipeline

Imports

In [0]:
%pip install openpyxl
import pandas as pd
from pyspark.sql import functions as F

Looking in indexes: [REDACTED]
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


Reading Source Files

In [0]:
base_path = "/Volumes/workspace/default/pei_assessment"

orders_df = spark.read.json(f"{base_path}/Orders.json", multiLine=True)
customers_df = spark.createDataFrame(pd.read_excel(f"{base_path}/Customer.xlsx", dtype={"phone": str}))
#products_df = spark.read.csv(f"{base_path}/Products.csv", header=True, inferSchema=True)
products_df = spark.read \
    .option("header", True) \
    .option("inferSchema", True) \
    .option("quote", '"') \
    .option("escape", '"') \
    .option("multiLine", True) \
    .csv(f"{base_path}/Products.csv")

print("orders_df:", orders_df.count(), "rows")
print("customers_df:", customers_df.count(), "rows")
print("products_df:", products_df.count(), "rows")

orders_df: 9994 rows
customers_df: 793 rows
products_df: 1851 rows


Writing Raw tables as Parquet as the columns name have " " and Delta doesnt support it.

In [0]:
orders_df.write.mode("overwrite").parquet(f"{base_path}/raw_tables/raw_orders")
customers_df.write.mode("overwrite").parquet(f"{base_path}/raw_tables/raw_customers")
products_df.write.mode("overwrite").parquet(f"{base_path}/raw_tables/raw_products")


Enriched Customer Table after cleaning the Customer's Name

In [0]:
customer_enriched_df = customers_df.withColumn(
    "Customer Name",
    F.trim(
        F.regexp_replace(
            F.regexp_replace(
                F.col("Customer Name"),
                r"[^a-zA-Z\s]",
                ""
            ),
            r"\s+",
            " "
        )
    )
)
customer_enriched_df.show(5)

+-----------+-------------+--------------------+--------------------+--------------------+-----------+-------------+----------------+----------+-----------+-------+
|Customer ID|Customer Name|               email|               phone|             address|    Segment|      Country|            City|     State|Postal Code| Region|
+-----------+-------------+--------------------+--------------------+--------------------+-----------+-------------+----------------+----------+-----------+-------+
|   PW-19240| Pierre Wener|bettysullivan808@...|   421.580.0902x9815|001 Jones Ridges ...|   Consumer|United States|      Louisville|  Colorado|      80027|   West|
|   GH-14410|  Gary Hansen|austindyer948@gma...|001-542-415-0246x314|00347 Murphy Unio...|Home Office|United States|         Chicago|  Illinois|      60653|Central|
|   KL-16555|Kelly Lampkin|clarencehughes280...|          7185624866|007 Adams Lane Su...|  Corporate|United States|Colorado Springs|  Colorado|      80906|   West|
|   AH-100

Enriched Product Table (Didnt drop the duplicates as the product name and other columns were unique)

In [0]:
product_enriched_df = products_df
product_enriched_df.show(5)

+---------------+---------------+------------+--------------------+--------+-----------------+
|     Product ID|       Category|Sub-Category|        Product Name|   State|Price per product|
+---------------+---------------+------------+--------------------+--------+-----------------+
|FUR-CH-10002961|      Furniture|      Chairs|Leather Task Chai...|New York|           81.882|
|TEC-AC-10004659|     Technology| Accessories|Imation Secure+ H...|Oklahoma|            72.99|
|OFF-BI-10002824|Office Supplies|     Binders|Recycled Easel Ri...|Colorado|             4.25|
|OFF-PA-10003349|Office Supplies|       Paper|          Xerox 1957| Florida|            5.184|
|TEC-AC-10003023|     Technology| Accessories|Logitech G105 Gam...|    Ohio|           47.496|
+---------------+---------------+------------+--------------------+--------+-----------------+
only showing top 5 rows


Created a lookup for product table post dropping the duplicates for creating enriched order.
 

In [0]:
product_lookup_df = product_enriched_df.select(
    "Product ID",
    "Category",
    "Sub-Category"
).dropDuplicates(["Product ID"])

Created Enriched Order Table as per requirment.

In [0]:
enriched_orders_df = orders_df \
    .join(customer_enriched_df.select(
            "Customer ID",
            "Customer Name",
            "Country"
        ),
        on="Customer ID",
        how="left"
    ) .join(
        product_lookup_df,
        on="Product ID",
        how="left"
    ).withColumn(
        "Profit",
        F.round(F.col("Profit"), 2)
    )

In [0]:
Writing all Enriched table as Parquet Tables.

  File <command-5037683329679634>, line 1
    Writing all Enriched table as Parquet Tables.
            ^
SyntaxError: invalid syntax


In [0]:
customer_enriched_df.write.mode("overwrite").parquet(f"{base_path}/enriched_tables/enriched_customers")
product_enriched_df.write.mode("overwrite").parquet(f"{base_path}/enriched_tables/enriched_products")
enriched_orders_df.write.mode("overwrite").parquet(f"{base_path}/enriched_tables/enriched_orders")

Creation of Aggregated Table

In [0]:
orders_with_year_df = enriched_orders_df.withColumn("Order Date",F.to_date(F.col("Order Date"), "d/M/yyyy")
    ).withColumn("Year",F.year(F.col("Order Date")))

In [0]:
aggregate_df = orders_with_year_df \
    .groupBy(
        "Year","Category","Sub-Category","Customer ID","Customer Name"
    ).agg(F.round(F.sum("Profit"), 2).alias("Total Profit"))

aggregate_df.show(5)

+----+---------------+------------+-----------+---------------+------------+
|Year|       Category|Sub-Category|Customer ID|  Customer Name|Total Profit|
+----+---------------+------------+-----------+---------------+------------+
|2017|Office Supplies|         Art|   AJ-10780| Anthony Jacobs|       17.01|
|2014|      Furniture|      Tables|   BH-11710|Brosina Hoffman|       85.31|
|2014|Office Supplies|         Art|   CM-12445|    Chuck Magee|       10.63|
|2014|      Furniture| Furnishings|   TB-21595| Troy Blackwell|      -11.32|
|2017|     Technology|      Phones|   NG-18355|     Nat Gilpin|      -74.36|
+----+---------------+------------+-----------+---------------+------------+
only showing top 5 rows


In [0]:
aggregate_df.write.mode("overwrite").parquet(f"{base_path}/aggregated_table/aggregated_orders")

In [0]:
orders_with_year_df.createOrReplaceTempView("enriched_orders")

SQL Output for Profit By Year

In [0]:
profit_by_year_df = spark.sql("""
    SELECT
        Year,
        ROUND(SUM(Profit), 2) AS Total_Profit
    FROM enriched_orders
    GROUP BY Year
    ORDER BY Year
""")

profit_by_year_df.show(truncate=False)

+----+------------+
|Year|Total_Profit|
+----+------------+
|2014|39185.75    |
|2015|63073.09    |
|2016|65073.23    |
|2017|111084.96   |
+----+------------+



SQL Output for Profit by Year and Product Category

In [0]:
profit_by_year_category_df = spark.sql("""
    SELECT
        Year,
        Category,
        ROUND(SUM(Profit), 2) AS Total_Profit
    FROM enriched_orders
    GROUP BY Year, Category
    ORDER BY Year, Category
""")

profit_by_year_category_df.show(truncate=False)

+----+---------------+------------+
|Year|Category       |Total_Profit|
+----+---------------+------------+
|2014|NULL           |523.12      |
|2014|Furniture      |-5331.04    |
|2014|Office Supplies|22500.32    |
|2014|Technology     |21493.35    |
|2015|NULL           |583.16      |
|2015|Furniture      |3027.18     |
|2015|Office Supplies|24519.39    |
|2015|Technology     |34943.36    |
|2016|NULL           |404.44      |
|2016|Furniture      |6889.5      |
|2016|Office Supplies|34555.53    |
|2016|Technology     |23223.76    |
|2017|NULL           |488.51      |
|2017|Furniture      |3041.54     |
|2017|Office Supplies|44273.12    |
|2017|Technology     |63281.79    |
+----+---------------+------------+



Profit By Customer

In [0]:
profit_by_customer_df = spark.sql("""
    SELECT
        `Customer ID`,
        `Customer Name`,
        ROUND(SUM(Profit), 2) AS Total_Profit
    FROM enriched_orders
    GROUP BY `Customer ID`, `Customer Name`
    ORDER BY `Customer ID`
""")

profit_by_customer_df.show(20, truncate=False)

+-----------+--------------------+------------+
|Customer ID|Customer Name       |Total_Profit|
+-----------+--------------------+------------+
|AA-10315   |Alex Avila          |-362.91     |
|AA-10375   |Allen Armold        |277.4       |
|AA-10480   |Andrew Allen        |436.08      |
|AA-10645   |Anna Andreadi       |857.8       |
|AB-10015   |Aaron Bergman       |129.68      |
|AB-10060   |Adam Bellavance     |2054.6      |
|AB-10105   |Adrian Barton       |5444.97     |
|AB-10150   |Aimee Bixby         |313.66      |
|AB-10165   |Alan Barnes         |220.6       |
|AB-10255   |Alejandro Ballentine|264.57      |
|AB-10600   |Ann Blume           |-275.28     |
|AC-10420   |Alyssa Crouse       |-62.13      |
|AC-10450   |Amy Cox             |1366.5      |
|AC-10615   |Ann Chong           |298.61      |
|AC-10660   |Anna Chung          |-28.7       |
|AD-10180   |Alan Dominguez      |1869.92     |
|AF-10870   |Art Ferguson        |317.62      |
|AF-10885   |Art Foster          |-163.1

Profit By Customer and Year

In [0]:
profit_by_customer_year_df = spark.sql("""
    SELECT
        `Customer ID`,
        `Customer Name`,
        Year,
        ROUND(SUM(Profit), 2) AS Total_Profit
    FROM enriched_orders
    GROUP BY
        `Customer ID`,
        `Customer Name`,
        Year
    ORDER BY
        `Customer ID`,
        Year
""")

profit_by_customer_year_df.show(20, truncate=False)

+-----------+---------------+----+------------+
|Customer ID|Customer Name  |Year|Total_Profit|
+-----------+---------------+----+------------+
|AA-10315   |Alex Avila     |2014|280.68      |
|AA-10315   |Alex Avila     |2015|7.0         |
|AA-10315   |Alex Avila     |2016|-747.09     |
|AA-10315   |Alex Avila     |2017|96.5        |
|AA-10375   |Allen Armold   |2014|16.72       |
|AA-10375   |Allen Armold   |2015|69.03       |
|AA-10375   |Allen Armold   |2016|122.73      |
|AA-10375   |Allen Armold   |2017|68.92       |
|AA-10480   |Andrew Allen   |2014|9.89        |
|AA-10480   |Andrew Allen   |2016|420.75      |
|AA-10480   |Andrew Allen   |2017|5.44        |
|AA-10645   |Anna Andreadi  |2014|77.64       |
|AA-10645   |Anna Andreadi  |2015|439.31      |
|AA-10645   |Anna Andreadi  |2016|334.63      |
|AA-10645   |Anna Andreadi  |2017|6.22        |
|AB-10015   |Aaron Bergman  |2014|12.82       |
|AB-10015   |Aaron Bergman  |2016|116.86      |
|AB-10060   |Adam Bellavance|2015|8.71  

Test cases for all the Tables

In [0]:
import unittest

class TestCustomerEnrichment(unittest.TestCase):

    def test_customer_row_count(self):
        self.assertEqual(customer_enriched_df.count(), 793)

    def test_customer_id_uniqueness(self):
        duplicate_count = (
            customer_enriched_df.count()
            - customer_enriched_df.select("Customer ID").distinct().count()
        )
        self.assertEqual(duplicate_count, 0)

    def test_customer_columns(self):
        self.assertEqual(len(customer_enriched_df.columns), 11)

    def test_null_customer_ids(self):
        null_count = customer_enriched_df.filter(
            F.col("Customer ID").isNull()
        ).count()
        self.assertEqual(null_count, 0)


suite = unittest.TestLoader().loadTestsFromTestCase(TestCustomerEnrichment)
unittest.TextTestRunner(verbosity=2).run(suite)

test_customer_columns (__main__.TestCustomerEnrichment.test_customer_columns) ... ok
test_customer_id_uniqueness (__main__.TestCustomerEnrichment.test_customer_id_uniqueness) ... ok
test_customer_row_count (__main__.TestCustomerEnrichment.test_customer_row_count) ... ok
test_null_customer_ids (__main__.TestCustomerEnrichment.test_null_customer_ids) ... ok

----------------------------------------------------------------------
Ran 4 tests in 0.978s

OK


<unittest.runner.TextTestResult run=4 errors=0 failures=0>

In [0]:
class TestProductEnrichment(unittest.TestCase):

    def test_product_row_count(self):
        self.assertEqual(product_enriched_df.count(), 1851)

    def test_product_columns(self):
        self.assertEqual(len(product_enriched_df.columns), 6)

    def test_product_id_nulls(self):
        null_count = product_enriched_df.filter(
            F.col("Product ID").isNull()
        ).count()
        self.assertEqual(null_count, 0)

    def test_product_price_data_type(self):
        price_type = dict(product_enriched_df.dtypes)["Price per product"]
        self.assertEqual(price_type, "double")

    def test_product_lookup_uniqueness(self):
        duplicate_count = (
            product_lookup_df.count()
            - product_lookup_df.select("Product ID").distinct().count()
        )
        self.assertEqual(duplicate_count, 0)

    def test_product_lookup_count(self):
        self.assertEqual(product_lookup_df.count(), 1818)


suite = unittest.TestLoader().loadTestsFromTestCase(TestProductEnrichment)
unittest.TextTestRunner(verbosity=2).run(suite)

test_product_columns (__main__.TestProductEnrichment.test_product_columns) ... ok
test_product_id_nulls (__main__.TestProductEnrichment.test_product_id_nulls) ... ok
test_product_lookup_count (__main__.TestProductEnrichment.test_product_lookup_count) ... ok
test_product_lookup_uniqueness (__main__.TestProductEnrichment.test_product_lookup_uniqueness) ... ok
test_product_price_data_type (__main__.TestProductEnrichment.test_product_price_data_type) ... ok
test_product_row_count (__main__.TestProductEnrichment.test_product_row_count) ... ok

----------------------------------------------------------------------
Ran 6 tests in 1.854s

OK


<unittest.runner.TextTestResult run=6 errors=0 failures=0>

In [0]:
class TestEnrichedOrders(unittest.TestCase):

    def test_order_row_count(self):
        self.assertEqual(enriched_orders_df.count(), 9994)

    def test_order_column_count(self):
        self.assertEqual(len(enriched_orders_df.columns), 15)

    def test_customer_columns_exist(self):
        self.assertIn("Customer Name", enriched_orders_df.columns)
        self.assertIn("Country", enriched_orders_df.columns)

    def test_product_columns_exist(self):
        self.assertIn("Category", enriched_orders_df.columns)
        self.assertIn("Sub-Category", enriched_orders_df.columns)

    def test_profit_rounding(self):
        invalid_count = enriched_orders_df.filter(
            F.col("Profit") != F.round(F.col("Profit"), 2)
        ).count()
        self.assertEqual(invalid_count, 0)

    def test_customer_unmatched_records(self):
        null_count = enriched_orders_df.filter(
            F.col("Customer Name").isNull()
        ).count()
        self.assertEqual(null_count, 137)

    def test_product_unmatched_records(self):
        null_count = enriched_orders_df.filter(
            F.col("Category").isNull()
        ).count()
        self.assertEqual(null_count, 204)


suite = unittest.TestLoader().loadTestsFromTestCase(TestEnrichedOrders)
unittest.TextTestRunner(verbosity=2).run(suite)

test_customer_columns_exist (__main__.TestEnrichedOrders.test_customer_columns_exist) ... ok
test_customer_unmatched_records (__main__.TestEnrichedOrders.test_customer_unmatched_records) ... ok
test_order_column_count (__main__.TestEnrichedOrders.test_order_column_count) ... ok
test_order_row_count (__main__.TestEnrichedOrders.test_order_row_count) ... ok
test_product_columns_exist (__main__.TestEnrichedOrders.test_product_columns_exist) ... ok
test_product_unmatched_records (__main__.TestEnrichedOrders.test_product_unmatched_records) ... ok
test_profit_rounding (__main__.TestEnrichedOrders.test_profit_rounding) ... ok

----------------------------------------------------------------------
Ran 7 tests in 2.939s

OK


<unittest.runner.TextTestResult run=7 errors=0 failures=0>

In [0]:
class TestAggregateTable(unittest.TestCase):

    def test_aggregate_columns(self):
        expected_columns = [
            "Year",
            "Category",
            "Sub-Category",
            "Customer ID",
            "Customer Name",
            "Total Profit"
        ]

        self.assertEqual(
            aggregate_df.columns,
            expected_columns
        )

    def test_aggregate_row_count(self):
        self.assertEqual(aggregate_df.count(), 8129)

    def test_no_duplicate_group_combinations(self):
        duplicate_count = (
            aggregate_df.count()
            - aggregate_df.dropDuplicates([
                "Year",
                "Category",
                "Sub-Category",
                "Customer ID",
                "Customer Name"
            ]).count()
        )

        self.assertEqual(duplicate_count, 0)

    def test_total_profit_reconciliation(self):
        source_profit = orders_with_year_df.agg(
            F.round(F.sum("Profit"), 2)
        ).first()[0]

        aggregate_profit = aggregate_df.agg(
            F.round(F.sum("Total Profit"), 2)
        ).first()[0]

        self.assertEqual(source_profit, aggregate_profit)


suite = unittest.TestLoader().loadTestsFromTestCase(TestAggregateTable)
unittest.TextTestRunner(verbosity=2).run(suite)

test_aggregate_columns (__main__.TestAggregateTable.test_aggregate_columns) ... ok
test_aggregate_row_count (__main__.TestAggregateTable.test_aggregate_row_count) ... ok
test_no_duplicate_group_combinations (__main__.TestAggregateTable.test_no_duplicate_group_combinations) ... ok
test_total_profit_reconciliation (__main__.TestAggregateTable.test_total_profit_reconciliation) ... ok

----------------------------------------------------------------------
Ran 4 tests in 3.031s

OK


<unittest.runner.TextTestResult run=4 errors=0 failures=0>